In [ ]:
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd

import os
import matplotlib.pyplot as plt
from glob import glob
import seaborn as sns 
from tqdm.notebook import tqdm

In [ ]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [ ]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [ ]:
moldf.shape

In [ ]:
import re

def process_list(data_list):
    """
    Processes a list of strings to perform the following operations:
    1.  Strips trailing newline characters.
    2.  Parses the 'LUCJ' string into three separate elements.
    3.  Converts the final element from a string to a float.
    """
    processed_list = []
    for item in data_list:
        item = item.strip()

        if "LUCJ" in item:
            # Use a regular expression to extract the components
            match = re.search(r'(LUCJ)\(L=(.*?)\)/(.*)', item)
            if match:
                processed_list.extend(match.groups())
            else:
                processed_list.append(item)
        else:
            processed_list.append(item)
    
    # Convert the last element to a float
    # We use a try-except block in case the last element is not a number
    try:
        processed_list[-1] = float(processed_list[-1])
    except (ValueError, IndexError):
        pass # The last element is not a float, so we ignore it

    return processed_list

In [ ]:
postprocessed = []
for i in tqdm(glob("./energies/*txt"),desc='Running'):
    with open(i,'r') as f:
        lines = f.readlines()

            
    moldict = pd.DataFrame.from_dict(dict(zip(["Basis Set","Molecule","Method","L","Injected","Energy"],process_list(lines))),orient='index').T
    postprocessed.append(moldict)

In [ ]:
LUCJDF=pd.concat(postprocessed).reset_index().drop(columns=['index']).astype({"L":int,"Energy":float})

In [ ]:
for a,b in moldf[['molecule','mol_filename']].values:
    if 'GDB' in b:
        name = b.replace('.xyz','')
        energyDF['Molecule'] = energyDF['Molecule'].replace(name,a)

In [ ]:
energyDF['L'] = len(energyDF)*[np.nan]
energyDF['Injected'] = len(energyDF)*[np.nan]

In [ ]:
upDF = pd.concat([energyDF,LUCJDF]).reset_index().drop(columns=['index'])

In [ ]:
uniqueBasis = energyDF['Basis Set'].unique()
uniqueMol = energyDF['Molecule'].unique()
uniqueInj = energyDF['Injected'].unique()
uniqueLayers = upDF['L'].unique()


In [ ]:
len(uniqueBasis) , len(uniqueMol)

In [ ]:
for basis in uniqueBasis:
    for mol in uniqueMol:
        submoldf = upDF[(upDF['Basis Set']==basis)&(upDF['Molecule']==mol)]
        submoldf['Deviation'] = len(submoldf) * [np.nan]
        submoldf.loc[:,'Deviation']=submoldf.loc[:,'Energy']-submoldf.loc[submoldf['Method']=="CASCI",'Energy'].values[0]
        print(submoldf)

In [ ]:


g = sns.FacetGrid(LUCJDF, col="Basis Set",row='Molecule')
g.map_dataframe(sns.barplot,x='Injected',y='Energy',hue="L",palette="Paired")
g.add_legend()

In [ ]:
g = sns.FacetGrid(upDF, col="Basis Set",row='Molecule')
g.map_dataframe(sns.barplot, x="Method",y='Energy',palette="Paired")

In [ ]:
# This creates a boolean mask that is True for every row where 'Method' contains "LUCJ"
mask = upDF['Method'].str.contains("LUCJ", case=False, na=False)

# Use the boolean mask to filter the DataFrame
filtered_df = upDF[mask]


In [ ]:
filtered_df

In [ ]:

devDF = upDF[(upDF['Basis Set']=='STO-3G')&(upDF['Molecule']=='ammonia')]
devDF['Deviation'] = (devDF['Energy'] - devDF.loc[devDF['Method'] == 'CASCI', 'Energy'].values[0])*1e3

In [ ]:
devDF

In [ ]:
sns.barplot(devDF,x='Method',y='Deviation')
# plt.yscale('symlog')
# plt.ylim(-1e-2,1e2)
